# NovaAI QLoRA Fine-Tuning Pipeline

Fine-tune `Qwen/Qwen2.5-1.5B-Instruct` with 4-bit QLoRA, then compare it with the normal FP16 base model on 70 held-out prompts. This notebook is designed for a **Google Colab Free Tesla T4** runtime and runs without API keys or paid services.

> NovaAI is fictional and was created only for this experiment.

Run the notebook from top to bottom. Generated outputs are real runtime artifacts; this notebook contains no prefilled metrics.

## 1. Check CUDA and the detected GPU

In Colab, select **Runtime → Change runtime type → T4 GPU** before continuing.

In [ ]:
import platform
import subprocess
import sys

import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab, choose a T4 GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({gpu_memory_gib:.1f} GiB)")
if "T4" not in gpu_name.upper():
    print("Note: this pipeline targets a Tesla T4; another CUDA GPU was detected.")
subprocess.run(["nvidia-smi"], check=False)

## 2. Install and verify dependencies

The installation happens before importing Transformers, TRL, PEFT, or bitsandbytes to avoid stale imports. PyTorch is supplied by Colab.

In [ ]:
from pathlib import Path

def find_requirements():
    candidates = [
        Path.cwd() / "requirements.txt",
        Path.cwd().parent / "requirements.txt",
        Path("/content/LoRA-Fine-Tuning-Pipeline/requirements.txt"),
    ]
    return next((path for path in candidates if path.is_file()), None)

requirements_path = find_requirements()
if requirements_path:
    command = [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)]
else:
    # Same versions as requirements.txt, for a notebook uploaded directly to Colab.
    packages = [
        "transformers==4.57.1", "datasets==4.4.1", "peft==0.17.1",
        "trl==0.24.0", "bitsandbytes==0.48.1", "accelerate==1.11.0",
        "pandas>=2.0,<3", "PyYAML>=6.0,<7", "sentencepiece>=0.2,<1",
    ]
    command = [sys.executable, "-m", "pip", "install", "-q", *packages]

print("Installing training dependencies...")
subprocess.check_call(command)
print("Dependency installation complete.")

In [ ]:
from importlib.metadata import version

import accelerate
import bitsandbytes as bnb
import datasets
import peft
import transformers
import trl

for package in ["torch", "transformers", "datasets", "peft", "trl", "bitsandbytes", "accelerate"]:
    print(f"{package}: {version(package)}")
assert torch.cuda.is_available(), "CUDA became unavailable after installation."
print("bitsandbytes import and CUDA prerequisite verified.")

## 3. Locate the repository and validate the datasets

Clone the full repository for the recommended workflow. If only the dataset files are missing, set `UPLOAD_DATA_IF_MISSING = True`; Colab will request the three JSONL files and `dataset_summary.json`.

In [ ]:
import json
from pathlib import Path

def discover_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/LoRA-Fine-Tuning-Pipeline"),
    ]
    for candidate in candidates:
        if (candidate / "src/dataset_utils.py").is_file() and (candidate / "configs/training_config.yaml").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository source files were not found. Clone the full repository, cd into it, "
        "and reopen notebooks/fine_tuning_colab.ipynb."
    )

PROJECT_ROOT = discover_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

required_data = ["train.jsonl", "validation.jsonl", "test.jsonl", "dataset_summary.json"]
missing_data = [name for name in required_data if not (DATA_DIR / name).is_file()]
UPLOAD_DATA_IF_MISSING = False

if missing_data and UPLOAD_DATA_IF_MISSING:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Optional uploads are available only in Google Colab.") from exc
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Upload these files: {missing_data}")
    uploaded = files.upload()
    for name in missing_data:
        if name not in uploaded:
            raise FileNotFoundError(f"Upload did not include {name}.")
        (DATA_DIR / name).write_bytes(uploaded[name])
    missing_data = [name for name in required_data if not (DATA_DIR / name).is_file()]

if missing_data:
    raise FileNotFoundError(
        f"Missing dataset files in {DATA_DIR}: {missing_data}. "
        "Clone the complete repository or enable UPLOAD_DATA_IF_MISSING."
    )

In [ ]:
import sys

import yaml

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dataset_utils import dataset_report, load_and_validate_splits

with (PROJECT_ROOT / "configs/training_config.yaml").open(encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

splits = load_and_validate_splits(DATA_DIR)
computed_report = dataset_report(splits)
with (DATA_DIR / "dataset_summary.json").open(encoding="utf-8") as handle:
    supplied_summary = json.load(handle)

assert computed_report["split_sizes"] == supplied_summary["splits"]
assert computed_report["unique_facts"] == supplied_summary["facts"]
assert computed_report["test_categories"] == supplied_summary["categories"]
print(json.dumps(computed_report, indent=2))
print("All dataset schema, size, uniqueness, and cross-split fact checks passed.")

## 4. Load the tokenizer and convert to Qwen chat format

TRL receives conversational `prompt` and `completion` fields. It applies Qwen's native chat template and computes loss only on completion tokens.

In [ ]:
from transformers import AutoTokenizer

from dataset_utils import render_training_example, to_conversational_sft_dataset

MODEL_ID = config["model"]["base_model_id"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_dataset = to_conversational_sft_dataset(splits["train"])
validation_dataset = to_conversational_sft_dataset(splits["validation"])
print(train_dataset)
print(validation_dataset)
print("\nRendered first training example:\n")
print(render_training_example(splits["train"][0], tokenizer))

## 5. Baseline evaluation with the normal FP16 model

This stage loads the unquantized base model in FP16, runs all 70 test prompts with deterministic greedy decoding, and saves the results before any training occurs.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
base_model.eval()
print(f"Loaded normal FP16 baseline: {MODEL_ID}")

In [ ]:
import pandas as pd

from inference import generate_predictions

base_results = generate_predictions(
    model=base_model,
    tokenizer=tokenizer,
    records=splits["test"],
    output_path=RESULTS_DIR / "base_model_results.csv",
    response_column="base_model_response",
    batch_size=config["inference"]["batch_size"],
    max_new_tokens=config["inference"]["max_new_tokens"],
    seed=config["seed"],
)
assert len(base_results) == 70
base_results.head(3)

In [ ]:
import gc

del base_model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
free_gib, total_gib = (value / 1024**3 for value in torch.cuda.mem_get_info())
print(f"Baseline model released. GPU memory free: {free_gib:.1f}/{total_gib:.1f} GiB")

## 6. Load Qwen in 4-bit and attach LoRA adapters

The T4-compatible quantization compute dtype is explicitly `float16`; BF16 remains disabled.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

target_modules = config["lora"]["target_modules"]
available_suffixes = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
missing_targets = sorted(set(target_modules) - available_suffixes)
if missing_targets:
    raise ValueError(f"LoRA target modules missing from this model: {missing_targets}")

lora_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["lora_alpha"],
    lora_dropout=config["lora"]["lora_dropout"],
    bias=config["lora"]["bias"],
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
assert model.is_loaded_in_4bit
assert quantization_config.bnb_4bit_compute_dtype == torch.float16

## 7. Fine-tune with TRL `SFTTrainer`

Validation and checkpoint saving occur after every epoch. Set `RESUME_FROM_LAST_CHECKPOINT` to `True` to resume an interrupted run when a checkpoint is still present on disk.

In [ ]:
from trl import SFTConfig, SFTTrainer

training = config["training"]
OUTPUT_DIR = PROJECT_ROOT / training["output_dir"]
ADAPTER_DIR = PROJECT_ROOT / training["adapter_dir"]
tokenizer.padding_side = "right"

sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=training["num_train_epochs"],
    per_device_train_batch_size=training["per_device_train_batch_size"],
    per_device_eval_batch_size=training["per_device_eval_batch_size"],
    gradient_accumulation_steps=training["gradient_accumulation_steps"],
    learning_rate=training["learning_rate"],
    max_length=config["model"]["max_length"],
    fp16=True,
    bf16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=training["logging_steps"],
    save_total_limit=training["save_total_limit"],
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    seed=config["seed"],
    data_seed=config["seed"],
    completion_only_loss=True,
    packing=False,
)
assert sft_config.fp16 is True and sft_config.bf16 is False

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)
print(f"Training output: {OUTPUT_DIR}")

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

RESUME_FROM_LAST_CHECKPOINT = True
last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.is_dir() else None
resume_checkpoint = last_checkpoint if RESUME_FROM_LAST_CHECKPOINT else None
print(f"Resume checkpoint: {resume_checkpoint}")
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(ADAPTER_DIR)
trainer.save_state()
print(f"LoRA adapter saved to {ADAPTER_DIR}")

## 8. Fine-tuned evaluation on the same 70 prompts

The next cell can restore the saved adapter after a runtime restart, provided dependencies, paths, config, tokenizer, and validated splits have been reloaded by rerunning Sections 2–4.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

ADAPTER_DIR = PROJECT_ROOT / config["training"]["adapter_dir"]
if not ADAPTER_DIR.is_dir():
    raise FileNotFoundError(f"Saved adapter not found at {ADAPTER_DIR}; complete training first.")

# Reuse the trained model when available; otherwise reconstruct it from disk.
if "model" not in globals():
    restore_quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    restored_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=restore_quantization,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )
    model = PeftModel.from_pretrained(restored_base, ADAPTER_DIR)

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()
print(f"Fine-tuned adapter ready: {ADAPTER_DIR}")

In [ ]:
import pandas as pd

from inference import generate_predictions

fine_tuned_results = generate_predictions(
    model=model,
    tokenizer=tokenizer,
    records=splits["test"],
    output_path=RESULTS_DIR / "fine_tuned_results.csv",
    response_column="fine_tuned_response",
    batch_size=config["inference"]["batch_size"],
    max_new_tokens=config["inference"]["max_new_tokens"],
    seed=config["seed"],
)
saved_base_results = pd.read_csv(RESULTS_DIR / "base_model_results.csv")
assert len(fine_tuned_results) == len(saved_base_results) == 70
fine_tuned_results.head(3)

## 9. Compare outputs and save reproducible metrics

Metrics are created only now, after both real prediction files exist and their ordered test identities match.

In [ ]:
import pandas as pd

from evaluation import METRIC_NAMES, compare_prediction_files

comparison, metrics = compare_prediction_files(
    base_results_path=RESULTS_DIR / "base_model_results.csv",
    fine_tuned_results_path=RESULTS_DIR / "fine_tuned_results.csv",
    comparison_path=RESULTS_DIR / "comparison.csv",
    metrics_path=RESULTS_DIR / "metrics.json",
    base_model_id=MODEL_ID,
    adapter_path=str(ADAPTER_DIR.relative_to(PROJECT_ROOT)),
)

summary_table = pd.DataFrame(
    {
        "base_model": metrics["base_model"],
        "fine_tuned_model": metrics["fine_tuned_model"],
        "delta": metrics["delta_fine_tuned_minus_base"],
    }
).loc[METRIC_NAMES]
display(summary_table.style.format("{:.4f}"))
display(comparison[["fact_id", "instruction", "expected_response", "base_model_response", "fine_tuned_response"]].head(10))
print(f"Saved comparison and metrics for {metrics['num_examples']} test examples.")

## 10. Verify and download artifacts

All four result files should be non-empty. The adapter directory contains only the PEFT adapter and tokenizer—not a duplicate full base model.

In [ ]:
expected_results = [
    RESULTS_DIR / "base_model_results.csv",
    RESULTS_DIR / "fine_tuned_results.csv",
    RESULTS_DIR / "comparison.csv",
    RESULTS_DIR / "metrics.json",
]
for path in expected_results:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty artifact: {path}")
    print(f"{path.relative_to(PROJECT_ROOT)}: {path.stat().st_size:,} bytes")

print(f"Adapter files: {[path.name for path in sorted(ADAPTER_DIR.iterdir())]}")
print("Download results/ and the adapter directory before ending the Colab session.")